# Week 1 — Day 1 (Alternative): Webpage Summarizer with Ollama

Same project as the Day 1 OpenAI version, but pointed at a local Ollama instance running Llama 3.2.

**Pre-req:** install Ollama from https://ollama.com and run `ollama run llama3.2` once to pull the model.

**Trick used here:** the OpenAI Python client speaks the same protocol as Ollama, so we just point `OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')` and reuse the exact same code path.

In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
# Smoke test the local model before doing real work
message = "Hello, Llama! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": message}]
)
print(response.choices[0].message.content)

## Website scraper class

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
site = Website("https://anthropic.com")
print(site.title)
print(site.text[:500])

## System and user prompts

In [ ]:
system_prompt = (
    "You are an assistant that analyzes the contents of a website "
    "and provides a short summary, ignoring text that might be navigation related. "
    "Respond in markdown."
)

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}\n"
    user_prompt += (
        "The contents of this website is as follows; "
        "please provide a short summary of this website in markdown. "
        "If it includes news or announcements, then summarize these too.\n\n"
    )
    user_prompt += website.text
    return user_prompt

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Summarize

In [ ]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model="llama3.2",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://anthropic.com")

In [ ]:
display_summary("https://cnn.com")

## Notes

- This won't work on JS-rendered sites (React apps etc.) — use Selenium or Playwright if you need to handle those.
- CloudFront-protected sites will return 403 — would need a more sophisticated fetching layer.